# Fock-Resolved Black-Box SQR Inference — Reproducibility Notebook (v2)

This notebook reproduces the key results of the study end-to-end.  
All tunable parameters are exposed in **Section 0** below; change them and re-run the notebook to explore.

**Study directory:** `studies/fock_resolved_black_box_sqr_inference_v2/`  
**Run the full study script instead:** `cd scripts && python run_study.py`  
**Quick profile:** `STUDY_PROFILE=quick python run_study.py`

## Section 0 — Tunable Parameters

Edit the values in this cell to explore different regimes.

In [ ]:
# ── Physical model ──────────────────────────────────────────────────────────
import numpy as np
TWO_PI = 2.0 * np.pi

OMEGA_Q_HZ      = 6.150e9          # qubit frequency [Hz]
OMEGA_C_HZ      = 5.241e9          # cavity frequency [Hz]
ALPHA_HZ        = -255.0e6         # qubit anharmonicity [Hz]
CHI_HZ          = -2.84e6          # dispersive shift χ [Hz]
CHI_PRIME_HZ    = -21.0e3          # second-order dispersive shift χ' [Hz]
KERR_HZ         = -28.0e3          # Kerr nonlinearity [Hz]

# ── Cavity state ─────────────────────────────────────────────────────────────
N_ACTIVE        = 4                # number of Fock sectors to resolve

# Default sector populations and SQR rotation angles for analytic cases
POPULATIONS     = np.array([0.41, 0.27, 0.19, 0.13])   # must sum to 1
THETA           = np.array([np.pi/3, np.pi/2, 0.9*np.pi, np.pi/4])  # rotation angles
PHI             = np.array([0.0, np.pi/3, np.pi/2, -np.pi/4])       # rotation phases

# ── Measurement ──────────────────────────────────────────────────────────────
N_WAIT_TIMES    = 6                # number of dispersive wait times
N_DISPLACEMENTS = 4                # number of displacement amplitudes
WAIT_TIME_MAX_S = 1.4e-6           # maximum wait time [s]
DISP_ALPHA_MAX  = 1.5              # maximum displacement amplitude

# ── Shot counts for sweeps ───────────────────────────────────────────────────
SHOT_GRID       = [100, 500, 1_000, 3_000, 10_000]   # for benchmark sweep
BENCHMARK_SHOTS = 5_000            # shots for single inference run
COHERENCE_SHOTS = 3_000            # shots for coherence witness

# ── Part 1 single-qubit baseline ─────────────────────────────────────────────
P1_N_SHOTS      = 3_000           # shots per Pauli observable
P1_NOISE_SIGMA  = 0.02            # Gaussian readout noise σ

# ── Part 2B non-uniqueness ────────────────────────────────────────────────────
FULL_STATE_RESTARTS = 6           # random restarts for full-state MLE

# ── Part 2F coherence sweep ───────────────────────────────────────────────────
COHERENCE_FRACTIONS = [0.0, 0.1, 0.25, 0.5, 0.75, 1.0]  # Model-B mixing fraction f

# ── Reproducibility seed ─────────────────────────────────────────────────────
RANDOM_SEED     = 42

print("Parameters loaded.")
print(f"  N_ACTIVE={N_ACTIVE}, populations={POPULATIONS}, sum={POPULATIONS.sum():.4f}")

## Section 1 — Imports and Setup

In [ ]:
import sys
import json
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import qutip as qt

# ── Resolve paths ─────────────────────────────────────────────────────────────
NB_DIR   = Path().resolve()  # this file lives in scripts/
STUDY_DIR = NB_DIR.parent
DATA_DIR  = STUDY_DIR / "data"
FIG_DIR   = STUDY_DIR / "figures"

# Add scripts directory to sys.path so study_lib is importable
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))

import runtime_compat  # noqa — adds cqed_sim to sys.path
import study_lib as lib
from common import (
    CHI, CHI_PRIME, KERR, ALPHA, OMEGA_Q, OMEGA_C,
    DEFAULT_POPULATIONS, DEFAULT_THETA, DEFAULT_PHI,
    apply_plot_style, N_ACTIVE as _N_ACTIVE,
)

apply_plot_style()
rng = np.random.default_rng(RANDOM_SEED)
print("Imports OK. cqed_sim version:", end=" ")
import importlib.metadata
try:
    print(importlib.metadata.version("cqed_sim"))
except Exception:
    print("(not installed as a package)")

## Section 2 — Physical Model and Target State

In [ ]:
from cqed_sim.core import DispersiveTransmonCavityModel

model = DispersiveTransmonCavityModel(
    omega_q=TWO_PI * OMEGA_Q_HZ,
    omega_c=TWO_PI * OMEGA_C_HZ,
    alpha=TWO_PI * ALPHA_HZ,
    chi=TWO_PI * CHI_HZ,
    chi_prime=TWO_PI * CHI_PRIME_HZ,
    kerr=TWO_PI * KERR_HZ,
    n_levels_qubit=3,
    n_levels_cavity=N_ACTIVE + 4,
)
print("Model created.")
print(f"  χ/(2π) = {CHI_HZ:.3f} MHz,  χ'/(2π) = {CHI_PRIME_HZ*1e3:.1f} kHz")

# Single-qubit target state (Part 1 building block)
rho_target = lib.single_qubit_target_state()
print("Target qubit state |ψ⟩ = Rx(π/3)·Rz(π/4)|g⟩:")
print("  ρ =", rho_target)

## Section 3 — Part 1: Single-Qubit MLE Building Block

Demonstrates that the core MLE machinery recovers a known qubit state from Pauli expectation values with Gaussian noise, before applying it to the joint cQED problem.

In [ ]:
result_p1 = lib.run_single_qubit_baseline(
    n_shots=P1_N_SHOTS,
    noise_sigma=P1_NOISE_SIGMA,
    n_repeats=8,
    rng=rng,
)

print(f"MLE mean fidelity : {result_p1['mle_mean_fidelity']:.4f}")
print(f"MLE std fidelity  : {result_p1['mle_std_fidelity']:.4f}")
print(f"LS  mean fidelity : {result_p1['ls_mean_fidelity']:.4f}")

# Quick fidelity-vs-shots plot
fig, ax = plt.subplots(figsize=(5, 3.5))
shots_grid = SHOT_GRID
fids = []
for ns in shots_grid:
    r = lib.run_single_qubit_baseline(n_shots=ns, noise_sigma=P1_NOISE_SIGMA, n_repeats=4, rng=rng)
    fids.append(r['mle_mean_fidelity'])
ax.semilogx(shots_grid, fids, 'o-')
ax.axhline(0.995, ls='--', color='gray', label='0.995 threshold')
ax.set_xlabel('Shots per observable')
ax.set_ylabel('Bures fidelity')
ax.set_title('Part 1 — Single-qubit MLE fidelity vs. shots')
ax.legend()
plt.tight_layout()
plt.show()

## Section 4 — Part 2A: Protocol Identifiability Analysis

SVD analysis of the 3m × (2N+1) design matrix determines which protocols can recover the recoverable subspace.

In [ ]:
# Build measurement settings
wait_settings  = lib.measurement_settings_wait_only(N_ACTIVE)
disp_settings  = lib.measurement_settings_displacement_only(N_ACTIVE)
comb_settings  = lib.measurement_settings_combined(N_ACTIVE)

protocols = {
    "wait_only":        wait_settings,
    "displacement_only": disp_settings,
    "combined":         comb_settings,
}

print(f"{'Protocol':<25} {'Transverse rank':>15} {'Full rank':>10} {'Cond #':>12}")
print("-" * 66)
for name, settings in protocols.items():
    summary = lib.protocol_identifiability_summary(settings, N_ACTIVE)
    cond_str = f"{summary['condition_number']:.2f}" if np.isfinite(summary['condition_number']) else "∞"
    print(f"{name:<25} {summary['transverse_rank']:>15} {summary['full_rank']:>10} {cond_str:>12}")

print("\nExpected: wait_only rank=8 (identifiable); displacement_only rank=2 (uninformative)")

## Section 5 — Part 2B: Full-State MLE Non-Uniqueness

Attempting to recover the full {pₙ, ρ_q^(n)} from multi-restart MLE reveals multiple optima — demonstrating fundamental non-identifiability.

In [ ]:
# Build a reference analytic case
ref_cases = lib.build_ideal_reference_cases(N_ACTIVE)
ref_case = ref_cases[0]  # mixed-input case

noisy_rows = lib.noisy_joint_rows(ref_case, wait_settings, n_shots=BENCHMARK_SHOTS, rng=rng)

mle_result = lib.run_full_state_mle_attempt(
    noisy_rows, wait_settings, N_ACTIVE,
    n_restarts=FULL_STATE_RESTARTS, rng=rng
)

print(f"Objective span across {FULL_STATE_RESTARTS} restarts: {mle_result['objective_span']:.2f}")
print(f"Population σ by sector: {[f'{s:.3f}' for s in mle_result['pop_std_by_sector']]}")
print(f"Gauge family size: {len(lib.construct_exact_gauge_family(ref_case, N_ACTIVE))}")

fig, axes = plt.subplots(1, N_ACTIVE, figsize=(10, 3))
for n, ax in enumerate(axes):
    pops = [r['populations'][n] for r in mle_result['restarts']]
    ax.hist(pops, bins=min(6, FULL_STATE_RESTARTS), edgecolor='k')
    ax.axvline(POPULATIONS[n], color='r', ls='--', label='true')
    ax.set_title(f'Sector n={n}')
    ax.set_xlabel('Inferred pₙ')
axes[0].set_ylabel('Count')
axes[0].legend()
fig.suptitle('Part 2B — Population spread across MLE restarts')
plt.tight_layout()
plt.show()

## Section 6 — Part 2C: Recoverable-Subspace Inference

After demonstrating non-uniqueness, we focus on the recoverable subspace: weighted transverse amplitudes uₙ = pₙ(Xₙ + iYₙ) and Z_tot = Σₙ pₙZₙ.

In [ ]:
print(f"{'Protocol':<20} {'N=1000 RMSE':>14} {'N=10000 RMSE':>14}")
print("-" * 52)

for pname, psettings in protocols.items():
    rmses = []
    for n_shots in [1_000, 10_000]:
        noisy = lib.noisy_joint_rows(ref_case, psettings, n_shots=n_shots, rng=rng)
        fit = lib.infer_weighted_mle(noisy, psettings, N_ACTIVE)
        summary = lib.recoverable_error_summary(fit, ref_case)
        rmses.append(summary['rmse'])
    print(f"{pname:<20} {rmses[0]:>14.4f} {rmses[1]:>14.4f}")

print("\nExpected: displacement_only RMSE flat (~0.17); wait_only and combined improve with shots")

## Section 7 — Part 2D/E: Per-Sector Oracle Metrics

Using true pₙ as oracle, we normalise the inferred weighted Bloch vector to obtain ρ_q^(n) and compute per-sector fidelity and trace distance.

In [ ]:
# Exact (noiseless) inference
exact_rows = lib.deterministic_joint_rows(ref_case, wait_settings)
fit_exact  = lib.infer_weighted_mle(exact_rows, wait_settings, N_ACTIVE)

sector_metrics = lib.per_sector_metrics(fit_exact, ref_case['sectors'])

print(f"{'Sector':>8} {'Oracle fidelity':>17} {'Oracle trace dist':>18} {'Bloch error':>13}")
print("-" * 62)
for n, m in enumerate(sector_metrics):
    print(f"{n:>8} {m['oracle_fidelity']:>17.4f} {m['oracle_trace_distance']:>18.4f} {m['oracle_bloch_error']:>13.4f}")

### Per-Fock Bloch-Vector Comparison Figure

In [ ]:
fig, axes = plt.subplots(1, N_ACTIVE, figsize=(12, 3.5))
labels = ['X', 'Y', 'Z']

for n, ax in enumerate(axes):
    true_bv = np.array([ref_case['sectors'][n].x,
                        ref_case['sectors'][n].y,
                        ref_case['sectors'][n].z])
    inf_bv  = sector_metrics[n]['oracle_bloch_inferred']
    x = np.arange(3)
    ax.bar(x - 0.2, true_bv, 0.35, label='True', color='steelblue')
    ax.bar(x + 0.2, inf_bv,  0.35, label='Oracle inferred', color='coral')
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylim(-1.1, 1.1)
    ax.set_title(f'Sector n={n}  F={sector_metrics[n]["oracle_fidelity"]:.3f}')
    ax.axhline(0, color='k', lw=0.5)

axes[0].set_ylabel('Bloch component')
axes[-1].legend(loc='upper right')
fig.suptitle('Per-Fock Bloch-Vector Comparison (exact data, oracle normalisation)')
plt.tight_layout()
plt.show()

## Section 8 — Part 2F: Model-B Coherence Sweep

The diagonal-model fit residual grows monotonically as the state gains off-diagonal cavity coherences, acting as a coherence witness.

In [ ]:
sweep_states = lib.build_coherence_sweep_states(N_ACTIVE, fractions=COHERENCE_FRACTIONS)

residuals = []
for f, state in zip(COHERENCE_FRACTIONS, sweep_states):
    rows = lib.noisy_joint_rows(state, wait_settings, n_shots=COHERENCE_SHOTS, rng=rng)
    fit  = lib.infer_weighted_ls(rows, wait_settings, N_ACTIVE)
    residuals.append(fit['residual_rms'])
    print(f"  f={f:.2f}  residual_rms={fit['residual_rms']:.4f}")

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(COHERENCE_FRACTIONS, residuals, 'o-')
ax.set_xlabel('Coherence fraction f')
ax.set_ylabel('LS residual RMS')
ax.set_title('Part 2F — Coherence witness: residual vs. mixing fraction')
plt.tight_layout()
plt.show()

# Verify monotonicity
diffs = np.diff(residuals)
print(f"\nMonotonically increasing: {np.all(diffs >= 0)}")

## Section 9 — Part 3: Comparison Questions

Loading the pre-computed answers from the study run.

In [ ]:
results = json.loads((DATA_DIR / "study_results.json").read_text())
answers = results['comparison_answers']

questions = {
    'q1_wait_only_recovers_transverse': 'Q1: Wait-only recovers transverse subspace?',
    'q2_displacement_needed_for_pn_zn': 'Q2: Displacement needed for pₙ / Zₙ?',
    'q3_full_state_mle_fails':          'Q3: Full-state MLE fails (non-unique)?',
    'q4_coherence_witness_works':       'Q4: Coherence witness works?',
    'q5_useful_coherence_fraction':     'Q5: Useful up to coherence fraction:',
    'q6_recommended_protocol':          'Q6: Recommended protocol:',
}

for key, label in questions.items():
    print(f"{label:<50} {answers.get(key, 'N/A')}")

## Section 10 — Load and Display Pre-Generated Figures

In [ ]:
from IPython.display import Image, display

figure_files = [
    ("Single-qubit fidelity scaling",   "single_qubit_fidelity_scaling.png"),
    ("Protocol comparison",             "protocol_comparison.png"),
    ("Per-Fock Bloch comparison",       "per_fock_bloch_comparison.png"),
    ("Full-state non-uniqueness",       "full_state_nonuniqueness.png"),
    ("Coherence witness residuals",     "coherence_witness_residuals.png"),
    ("Coherence sweep",                 "coherence_sweep.png"),
    ("Pulse-case recovery",             "pulse_case_recovery.png"),
    ("Robustness sensitivity",          "robustness_sensitivity.png"),
]

for title, fname in figure_files:
    fpath = FIG_DIR / fname
    if fpath.exists():
        print(f"\n--- {title} ---")
        display(Image(filename=str(fpath), width=700))
    else:
        print(f"[missing] {fpath}")

## Section 11 — Validation Checks

Reproduces the 8 validation checks from `validate_results.py`.

In [ ]:
validation = json.loads((DATA_DIR / "validation_summary.json").read_text())

all_pass = True
print(f"{'Check':<50} {'Status':>8}")
print("-" * 62)
for check in validation['checks']:
    status = "PASS" if check['passed'] else "FAIL"
    if not check['passed']:
        all_pass = False
    print(f"{check['name']:<50} {status:>8}")

print()
print(f"Overall: {'ALL PASS' if all_pass else 'SOME FAILED'}  "
      f"({sum(c['passed'] for c in validation['checks'])}/{len(validation['checks'])} passed)")

## Section 12 — Re-Run Full Study (Optional)

Uncomment and run the cell below to re-execute the entire study from scratch using the current parameter values above.  
**Warning:** full profile takes ~105 s; use `STUDY_PROFILE=quick` for a ~30 s run.

In [ ]:
# import subprocess, os
# env = os.environ.copy()
# env["STUDY_PROFILE"] = "quick"   # change to "full" for complete run
# result = subprocess.run(
#     [sys.executable, str(NB_DIR / "run_study.py")],
#     cwd=str(NB_DIR),
#     capture_output=True, text=True, env=env
# )
# print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
# if result.returncode != 0:
#     print("STDERR:", result.stderr[-1000:])